In [1]:
import datetime
from pathlib import Path
from typing import Final, Literal

import pandas as pd
import pycaret.regression as pyc

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATASET = Literal["AGP", "GGMP"]

dataset:DATASET = "AGP"
# dataset:DATASET = "GGMP"

meta_path = PROJECT_ROOT / "datasets/processed" / dataset / "meta.tsv"
otu_path = PROJECT_ROOT / "datasets/processed" / dataset / "otu.tsv"

output_path = PROJECT_ROOT / "result" / dataset
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
def split_otu_by_health(
    meta_path: Path, otu_path: Path
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Read meta.tsv and otu.tsv
    meta_df = pd.read_csv(meta_path, sep="\t")
    meta_df = meta_df.set_index("id")

    otu_df = pd.read_csv(otu_path, sep="\t")
    otu_df = otu_df.set_index("id")

    # Split otu_df based on the 'health' column in meta_df
    healthy_otu_df = otu_df[meta_df["health"] == "y"]
    # nonhealthy_otu_df = otu_df[meta_df["health"] == "n"]

    predicted_age_df = pd.merge(
        healthy_otu_df, meta_df["age"], left_index=True, right_index=True, how="inner"
    )

    return predicted_age_df, meta_df, otu_df


# Split otu.tsv into healthy and get predicted age dataframe
predicted_age_df, meta_df, otu_df = split_otu_by_health(meta_path, otu_path)


In [ ]:
def model_health_ages(
    predicted_age_df: pd.DataFrame,
    otu_df: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:

    # Use pycaret to model healthy otu_df and predict the physiological age of the samples
    pyc.setup(
        data=predicted_age_df,
        target="age",
        session_id=123,
    )

    print("X Train Shape:", pyc.get_config("X_train").shape)
    print("X Test Shape:", pyc.get_config("X_test").shape)

    # Compare regression models and select the one with lowest MAE
    best_model = pyc.compare_models(
        sort="MAE",
        errors="raise",
    )

    compare_result = pyc.pull()
    compare_result.to_csv(
        output_dir / "compare_models.tsv",
        sep="\t",
        index=True,
    )

    # MAE of the original winning model
    best_mae = float(compare_result.iloc[0]["MAE"])

    # Tune the winning model
    tuned_model = pyc.tune_model(
        best_model,
        optimize="MAE",
        n_iter=50,
    )

    tune_result = pyc.pull()
    tune_result.to_csv(
        output_dir / "tuned_best_model.tsv",
        sep="\t",
        index=True,
    )

    # PyCaret tuning output contains a "Mean" row
    tuned_mae = float(tune_result.loc["Mean", "MAE"])

    print(f"Original best model MAE: {best_mae:.4f}")
    print(f"Tuned model MAE:        {tuned_mae:.4f}")

    # Only keep tuned model if tuning actually improved MAE
    if tuned_mae < best_mae:
        selected_model = tuned_model
        print("Using tuned model.")
    else:
        selected_model = best_model
        print("Tuning did not improve MAE. Using original best model.")

    # Refit selected model on all healthy training samples
    final_best_model = pyc.finalize_model(selected_model)

    # Predict gut age for all samples
    prediction_result = pyc.predict_model(
        final_best_model,
        data=otu_df,
    )

    age_predictions = prediction_result["prediction_label"]

    # Save final model
    current_date = datetime.datetime.now(datetime.UTC).strftime("%Y%m%d")

    pyc.save_model(
        final_best_model,
        str(output_dir / f"final_best_model_{current_date}"),
    )

    return age_predictions.to_frame()


# Model healthy otu dataframe and predict ages
age_predictions = model_health_ages(predicted_age_df, otu_df, output_path)

In [ ]:
def calculate_raw_gai(
    meta_df: pd.DataFrame, age_predictions: pd.DataFrame
) -> pd.DataFrame:
    """Add predicted age minus chronological age as the raw GAI."""
    meta_df["raw GAI"] = age_predictions["prediction_label"] - meta_df["age"]
    return meta_df

# Calculate raw GAI for all samples and add it to meta_df
meta_df = calculate_raw_gai(meta_df, age_predictions)

In [ ]:
AGE_RANGES: Final[tuple[tuple[int, int], ...]] = (
    (18, 20),
    (20, 25),
    (25, 30),
    (30, 35),
    (35, 40),
    (40, 45),
    (45, 50),
    (50, 55),
    (55, 60),
    (60, 65),
    (65, 70),
    (70, 75),
    (75, 100),
)


def calculate_adjust_value(
    meta_df: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    """Calculate healthy-cohort GAI adjustment values for each age range."""

    adjust_values: list[float] = []

    for start_age, end_age in AGE_RANGES:

        # Only healthy participants in this age range
        healthy_in_age_range = (
            (meta_df["health"] == "y")
            & (meta_df["age"] >= start_age)
            & (meta_df["age"] < end_age)
        )

        adjust_value = meta_df.loc[
            healthy_in_age_range,
            "raw GAI",
        ].mean()

        # Prevent silently generating NaNs if an age bin has no healthy samples
        if pd.isna(adjust_value):
            raise ValueError(
                f"No healthy samples found in age range "
                f"{start_age}-{end_age}."
            )

        adjust_values.append(adjust_value)

    # Save adjustment values
    adjust_df = pd.DataFrame(
        {
            "age_range": AGE_RANGES,
            "adjust_value": adjust_values,
        }
    )

    adjust_df.to_csv(
        output_dir / "adjust_values.tsv",
        sep="\t",
        index=False,
    )

    # Assign each adjustment value to everybody in that age range
    for (start_age, end_age), adjust_value in zip(
        AGE_RANGES,
        adjust_values,
        strict=True,
    ):

        in_age_range = (
            (meta_df["age"] >= start_age)
            & (meta_df["age"] < end_age)
        )

        meta_df.loc[
            in_age_range,
            "adjust value",
        ] = adjust_value

    return meta_df



# Calculate adjust values based on age ranges and add them to meta_df
meta_df = calculate_adjust_value(meta_df, output_path)

In [ ]:
def calculate_corrected_gai(meta_df):
    # Calculate corrected GAI by subtracting adjust value from raw GAI
    meta_df["corrected GAI"] = meta_df["raw GAI"] - meta_df["adjust value"]

    return meta_df


# Calculate corrected GAI and add it to meta_df
meta_df = calculate_corrected_gai(meta_df)

In [ ]:

def save_result(meta_df: pd.DataFrame, result_path: Path) -> None:
    """Save the completed results table as a TSV file."""
    meta_df.to_csv(result_path, sep="\t", index=True)
    print(f"Saved result as {result_path}")

# Save final result as result.tsv
save_result(meta_df, output_path / "result.tsv")
